In [ ]:

import requests
import pandas as pd
from io import BytesIO


class APIClient:
    """Client pour récupérer les données de Paris Open Data."""

    VELIB_URL = (
        "https://opendata.paris.fr/api/explore/v2.1/"
        "catalog/datasets/velib-disponibilite-en-temps-reel/"
        "exports/csv?use_labels=true"
    )
    METEO_URL = (
        "https://public-api.meteofrance.fr/public/DPObs/v2"
        "station/infrahoraire-6m?id_station=75&format=json"
    )

    def __init__(self, dataset, timeout: int = 30):
        self.timeout = timeout
        self.session = requests.Session()

        self.choose_dataset(dataset)

    def choose_dataset(self, dataset: str) -> None:
        """Choisit le dataset à récupérer."""
        if dataset == "velib":
            self.dataset_url = self.VELIB_URL
        elif dataset == "meteo":
            self.dataset_url = self.METEO_URL
        else:
            raise ValueError("Dataset inconnu. Choisissez 'velib' ou 'meteo'.")

    def get_dataset(self) -> pd.DataFrame:
        """Récupère le CSV et le convertit en DataFrame."""

        if self.dataset_url == self.VELIB_URL:
            response = self.session.get(
                self.dataset_url,
                timeout=self.timeout,
            )
        elif self.dataset_url == self.METEO_URL:
            response = self.session.get(
                self.dataset_url,
                timeout=self.timeout,
                apikey="YOUR_API_KEY"  # Remplacez par votre clé API
            )
        response.raise_for_status()

        # Le CSV de Velib est encodé en UTF-8, celui de la météo en ISO-8859-1
        if self.dataset_url == self.VELIB_URL:
            # Le CSV de la météo est encodé en ISO-8859-1
            return pd.read_csv(
                BytesIO(response.content),
                encoding="ISO-8859-1",
                sep=None,
                engine="python",
            )
        elif self.dataset_url == self.METEO_URL:
            # Le retour de la météo est un JSON, on le convertit en DataFrame
            return pd.DataFrame(response.json())



    def close(self) -> None:
        self.session.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

In [18]:
def main():
    with APIClient("meteo") as client:
        # Récupération des données
        data = client.get_dataset()
        print(data.head())

        # Étape suivante :
        # nettoyer et transformer les données reçues
        # velib_clean = ...
        # meteo_clean = ...


if __name__ == "__main__":
    main()


JSONDecodeError: Expecting value: line 1 column 1 (char 0)